In [1]:
cd ..

/Users/martinsevilla/Downloads/congestion-bogota


In [2]:
import os
import pandas as pd
import torch

from src.data import GraphTimeSeriesLoader, GNNTimeSeriesDataset

%load_ext autoreload
%autoreload 2

In [3]:
processed_data_dir = "data/datos_procesados"
dataset_filenames = {
    "buildings": f"{processed_data_dir}/buildings/roads_buildings.csv",
    "bus_stops": f"{processed_data_dir}/bus_stops/bus_stops_per_road.parquet",
    "events": f"{processed_data_dir}/events/events_dataset.csv",
    "incidents": f"{processed_data_dir}/incidents/incidents_dataset.csv",
    "irregularities": [f"{processed_data_dir}/irregularidades_procesadas/" + f for f in os.listdir(f"{processed_data_dir}/irregularidades_procesadas/") if f.endswith('.parquet')],
    # "coi_points": "pmts_processed/coi_puntos_dataset.parquet",
    # "coi_parks": "pmts_processed/coi_parques_dataset.parquet",
    "coi_points": [f"{processed_data_dir}/pmts_processed/coi_puntos/" + f for f in os.listdir(f"{processed_data_dir}/pmts_processed/coi_puntos/") if f.endswith('.parquet')],
    "coi_tracts": [f"{processed_data_dir}/pmts_processed/coi_tramos/" + f for f in os.listdir(f"{processed_data_dir}/pmts_processed/coi_tramos/") if f.endswith('.parquet')],
    "road_state": f"{processed_data_dir}/road_state/road_state_dateset.parquet", # TODO: Should be 'dataset' no 'dateset'
}
graph_filename = f"{processed_data_dir}/conectivity_graph/road_graph.pickle"

In [4]:
loader = GraphTimeSeriesLoader(
    graph_pkl_filename=graph_filename,
    buildings_csv_filename=dataset_filenames["buildings"],
    bus_stops_parquet_filename=dataset_filenames["bus_stops"],
    events_csv_filename=dataset_filenames["events"],
    incidents_csv_filename=dataset_filenames["incidents"],
    road_state_parquet_filename=dataset_filenames["road_state"],
    irregularities_filenames=dataset_filenames["irregularities"],
    coi_points_filenames=dataset_filenames["coi_points"],
    coi_tracts_filenames=dataset_filenames["coi_tracts"],
)

In [5]:
# Convert the nx graph to a pandas DataFrame to check that everything is loaded correctly
all_nodes_df = pd.DataFrame.from_dict(dict(loader.base_graph.nodes(data=True)), orient='index')
all_nodes_df

,id_mvi,MVICODIGO,MVICCALZAD,MVICIV,MVICCAT,MVITCLA,MVITIPO,MVINOMBRE,MVINALTERN,MVINPRINCI,...,warehouse,water_tank,water_tankmm,yes,yesmm,num_stops,num_stops_m,num_stops_s,mean_distance_meters,median_distance_meters
0_B.1,0_B.1,10464,0.0,0,17000274,6,CL,SIN_NMG,,022A000000,...,1.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0
0_B.2,0_B.2,56308,0.0,0,12UU0000,1,,SIN_NMG,,,...,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0
0_B.3,0_B.3,105670,0.0,0,10010189,3,CL,SIN_NMG,,064D000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0_B.4,0_B.4,136845,0.0,0,9000796,3,KR,SIN_NMG,,1120000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0_B.5,0_B.5,136456,0.0,0,10UU0000,6,,AVENIDA DEL CONGRESO EUCARISTICO,,,...,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9004787_B.1,9004787_B.1,116597,91014617.0,9004787,9001511,3,KR,SIN_NMG,,112B000000,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
91018094_N.1,91018094_N.1,105368,10011277.0,91018094,10005941,4,TV,SIN_NMG,,074B000000,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
91022147_B.1,91022147_B.1,140034,4007863.0,91022147,91022147,4,Null,,,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
91022525_B.1,91022525_B.1,106162,150890.0,91022525,5008911,3,CL,SIN_NMG,,055B0000S0,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# Example usage of the loader to load data for specific nodes and time windows
nodes_to_load = ["50008335_TF.3", "18001197_TF.1"]
time_windows_to_load = [
    # Query for 2 hours (8 * 15 mins)
    ("2024-12-05 07:30:00", 8),
    # Query for 1 hour (4 * 15 mins)
    ("2024-11-29 18:30:00", 4),
]

# Load the data
static_features, temporal_features = loader.load_data(
    nodes_to_load, time_windows_to_load
)


--- Starting data load for 2 nodes ---
--- Load complete. Found 2 static records and 4 temporal records. ---


In [7]:
static_features

,id_mvi,MVICODIGO,MVICCALZAD,MVICIV,MVICCAT,MVITCLA,MVITIPO,MVINOMBRE,MVINALTERN,MVINPRINCI,...,warehouse,water_tank,water_tankmm,yes,yesmm,num_stops,num_stops_m,num_stops_s,mean_distance_meters,median_distance_meters
node_id,,,,,,,,,,,,,,,,,,,,,
50008335_TF.3,50008335_TF.3,29511,508984.0,50008335,8007931,1,AK,AVENIDA BOYACA,,0720000000,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
18001197_TF.1,18001197_TF.1,35773,413389.0,18001197,18005545,2,CL,SIN_NMG,,02800000S0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
temporal_features

,road_id,timeframe,ACCIDENT_MAJOR_count,ACCIDENT_MAJOR_nThumbsUp,ACCIDENT_MINOR_count,ACCIDENT_MINOR_nThumbsUp,HAZARD_ON_ROAD_CAR_STOPPED_count,HAZARD_ON_ROAD_CAR_STOPPED_nThumbsUp,HAZARD_ON_ROAD_CONSTRUCTION_count,HAZARD_ON_ROAD_CONSTRUCTION_nThumbsUp,...,causeType_0,causeType_ACCIDENT,causeType_HAZARD,causeType_ROAD_CLOSED,trend,speed,jamLevel,severity,regularSpeed,source
0,18001197_TF.1,2024-11-29 18:30:00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0.849057,5.990566,4.0,5.0,10.125094,irregularities
1,18001197_TF.1,2024-11-29 18:45:00,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,1.000000,5.900000,4.0,5.0,10.190000,irregularities
2,50008335_TF.3,2024-12-05 07:45:00,0.790169,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,-0.735294,4.418824,4.0,5.0,7.457647,irregularities
3,50008335_TF.3,2024-12-05 08:00:00,0.197542,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0.000000,4.360000,4.0,5.0,7.590000,irregularities


In [9]:
# Converting it to a PyTorch Dataset for GNNs
samples = [(node, '2024-01-01 00:00:00') for node in loader.base_graph.nodes()]
gnn_dataset = GNNTimeSeriesDataset(loader, loader.base_graph, samples)

# Then we can use it with a standard PyTorch DataLoader
# TODO: We may need to implement a custom collate_fn for batching graph data
torch_dataloader = torch.utils.data.DataLoader(
    gnn_dataset,
    batch_size=32,
    shuffle=True,
)

for batch in torch_dataloader:
    # Training loop, `batch` contains tensors ready for the model
    pass


--- Starting data load for 7 nodes ---
--- Load complete. Found 7 static records and 7 temporal records. ---

--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records and 3 temporal records. ---

--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records and 3 temporal records. ---

--- Starting data load for 5 nodes ---
--- Load complete. Found 5 static records and 5 temporal records. ---

--- Starting data load for 6 nodes ---
--- Load complete. Found 6 static records and 6 temporal records. ---

--- Starting data load for 4 nodes ---
--- Load complete. Found 4 static records and 4 temporal records. ---

--- Starting data load for 5 nodes ---


KeyboardInterrupt: 